# Week 10 — Support Vector Machines & Kernel Methods
## Notebook 1: Linear Separability & Maximum Margin

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐ (Intermediate) |
| **Estimated Time** | 2.5 hours |
| **Dataset Theme** | Email Spam Classification |
| **Key Skills** | Hyperplane geometry, margin computation, support vectors |

---

## Why This Matters

Before we can understand Support Vector Machines, we need to understand two foundational ideas:

1. **Linear Separability** — when can a straight line (or hyperplane) perfectly divide two classes?
2. **Maximum Margin** — among all dividing lines, which one is the *best*?

These ideas are not just theoretical curiosities. The maximum margin principle is the geometric heart of SVMs, and it directly explains why SVMs often generalize better than other classifiers. Get this right, and the rest of SVMs will click into place.

---

## The Analogy First: Separating M&Ms with a Ruler

Imagine you have a table covered with red M&Ms (spam emails) and blue M&Ms (legitimate emails). You want to separate them using a ruler laid flat on the table.

- **Many rulers work.** You can tilt the ruler slightly left, slightly right, slightly toward you — and dozens of different positions all manage to keep red on one side and blue on the other.
- **Most rulers are risky.** If the ruler is very close to the blue pile, and you slide in one new M&M, it might land on the wrong side just because the ruler was a millimeter off.
- **The BEST ruler** is the one positioned exactly in the middle — equidistant from the closest red and the closest blue M&M. This ruler gives you the **maximum gap** (margin) between the two groups.

The **maximum margin hyperplane** is that best ruler. It's not just any separator — it's the one that maximizes safety for new, unseen data points.

> **Key insight:** A wider margin means more room for new test points to land correctly, even if they're slightly different from the training data. This is why maximum margin = better generalization.

## 1. Linear Separability: The Plain English Version

A dataset is **linearly separable** if you can draw a single straight line (in 2D), plane (in 3D), or hyperplane (in nD) that perfectly separates all points of class +1 from all points of class -1 — with **zero mistakes**.

### The Hyperplane Equation

In any number of dimensions, the decision boundary is:

$$\mathbf{w} \cdot \mathbf{x} + b = 0$$

Where:
- $\mathbf{w}$ = the **normal vector** to the hyperplane (points perpendicular to it)
- $b$ = the **bias** (shifts the hyperplane away from the origin)
- $\mathbf{x}$ = a data point

### Classification Rule

$$\hat{y} = \text{sign}(\mathbf{w} \cdot \mathbf{x} + b) = \begin{cases} +1 & \text{(spam)} \\ -1 & \text{(not spam)} \end{cases}$$

### Distance from a Point to the Hyperplane

$$d(\mathbf{x}_0) = \frac{|\mathbf{w} \cdot \mathbf{x}_0 + b|}{\|\mathbf{w}\|}$$

This formula tells us how far any point $\mathbf{x}_0$ is from the decision boundary. It is the key to computing margins.

### Margin Width

The **margin** is the total width of the gap between the two classes, measured perpendicular to the hyperplane:

$$\text{Margin} = \frac{2}{\|\mathbf{w}\|}$$

The factor of 2 comes from the convention that the two margin hyperplanes are $\mathbf{w} \cdot \mathbf{x} + b = +1$ and $\mathbf{w} \cdot \mathbf{x} + b = -1$. The perpendicular distance between them is exactly $2 / \|\mathbf{w}\|$.

In [ ]:
# ============================================================
# CELL 1: Imports and global settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Reproducibility and style — required for all notebooks in this series
np.random.seed(42)
sns.set_theme(style='whitegrid')

# Make plots larger and cleaner
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 13

print("Libraries loaded successfully.")
print(f"NumPy version : {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Creating the Email Spam Dataset

We'll work with a 2D dataset representing emails classified as spam or not spam.

**Features:**
- `word_count` — total number of words in the email (0–500)
- `spam_word_ratio` — fraction of words that are spam-indicator words like "free", "winner", "click" (0–1)

**Labels:**
- `+1` = spam (high word count AND high spam word ratio)
- `-1` = legitimate (low word count OR low spam word ratio)

We'll engineer this dataset to be **perfectly linearly separable** — a clean toy problem that lets us study the geometry without noise getting in the way.

In [ ]:
# ============================================================
# CELL 2: Generate 2D linearly separable email spam dataset
# ============================================================
np.random.seed(42)

n_spam = 100  # 100 spam emails
n_legit = 100  # 100 legitimate emails

# Spam emails: high word count (300-500) + high spam word ratio (0.55-0.95)
spam_word_count     = np.random.uniform(300, 500, n_spam)
spam_word_ratio     = np.random.uniform(0.55, 0.95, n_spam)

# Legitimate emails: low word count (0-200) + low spam word ratio (0.05-0.45)
legit_word_count    = np.random.uniform(0, 200, n_legit)
legit_word_ratio    = np.random.uniform(0.05, 0.45, n_legit)

# Combine into feature matrix X and label vector y
X_spam  = np.column_stack([spam_word_count, spam_word_ratio])
X_legit = np.column_stack([legit_word_count, legit_word_ratio])

X = np.vstack([X_spam, X_legit])          # shape: (200, 2)
y = np.hstack([np.ones(n_spam), -np.ones(n_legit)])  # +1 = spam, -1 = legit

# Store in a tidy DataFrame for inspection
df = pd.DataFrame(X, columns=['word_count', 'spam_word_ratio'])
df['label'] = y
df['class'] = df['label'].map({1.0: 'Spam', -1.0: 'Legitimate'})

print(f"Dataset shape        : {X.shape}")
print(f"Spam emails          : {n_spam}")
print(f"Legitimate emails    : {n_legit}")
print("\nFirst 5 rows:")
print(df.head())
print("\nClass distribution:")
print(df['class'].value_counts())

In [ ]:
# ============================================================
# CELL 3: Visualize the raw dataset
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7))

# Plot spam points (label +1)
spam_mask = df['label'] == 1
ax.scatter(
    df.loc[spam_mask, 'word_count'],
    df.loc[spam_mask, 'spam_word_ratio'],
    color='crimson', marker='o', s=50, alpha=0.7,
    label='Spam (+1)', edgecolors='darkred', linewidths=0.4
)

# Plot legitimate points (label -1)
legit_mask = df['label'] == -1
ax.scatter(
    df.loc[legit_mask, 'word_count'],
    df.loc[legit_mask, 'spam_word_ratio'],
    color='steelblue', marker='s', s=50, alpha=0.7,
    label='Legitimate (-1)', edgecolors='navy', linewidths=0.4
)

ax.set_xlabel('Word Count')
ax.set_ylabel('Spam Word Ratio')
ax.set_title('Email Spam Dataset\n(Linearly Separable — 200 Emails)')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("Observation: The two classes are clearly separated.")
print("There is a visible gap between spam (top-right) and legitimate (bottom-left).")

## 3. Many Valid Separating Hyperplanes

The key insight before SVMs: for any linearly separable dataset, there are **infinitely many** hyperplanes that correctly separate the classes. Each one is a valid classifier, but they are not all equally good.

We'll plot 5 different lines, all of which perfectly classify every email. The question is: **which one is best?**

We parameterize each line in the form:

$$\text{spam\_word\_ratio} = m \times \text{word\_count} + c$$

Or equivalently: $w_1 \cdot x_1 + w_2 \cdot x_2 + b = 0$.

In [ ]:
# ============================================================
# CELL 4: Plot 5 different valid separating hyperplanes
# ============================================================
# Each hyperplane is specified as (slope m, intercept c) in feature space
# Line equation: spam_word_ratio = m * word_count + c
# These were hand-chosen to all correctly separate the two classes
candidate_hyperplanes = [
    {'m': 0.00090, 'c': 0.27,  'label': 'Line A (steep slope)', 'color': 'purple'},
    {'m': 0.00080, 'c': 0.30,  'label': 'Line B (moderate)',    'color': 'orange'},
    {'m': 0.00070, 'c': 0.33,  'label': 'Line C (central)',     'color': 'green'},
    {'m': 0.00060, 'c': 0.38,  'label': 'Line D (gentle slope)','color': 'brown'},
    {'m': 0.00050, 'c': 0.42,  'label': 'Line E (flat slope)',  'color': 'magenta'},
]

x_range = np.linspace(-20, 520, 300)  # x-axis range for plotting lines

fig, ax = plt.subplots(figsize=(12, 8))

# Scatter data points
ax.scatter(df.loc[spam_mask, 'word_count'], df.loc[spam_mask, 'spam_word_ratio'],
           color='crimson', marker='o', s=50, alpha=0.6,
           label='Spam (+1)', edgecolors='darkred', linewidths=0.4, zorder=3)
ax.scatter(df.loc[legit_mask, 'word_count'], df.loc[legit_mask, 'spam_word_ratio'],
           color='steelblue', marker='s', s=50, alpha=0.6,
           label='Legitimate (-1)', edgecolors='navy', linewidths=0.4, zorder=3)

# Plot each candidate hyperplane
for hp in candidate_hyperplanes:
    y_line = hp['m'] * x_range + hp['c']
    ax.plot(x_range, y_line, color=hp['color'], linewidth=2,
            label=hp['label'], linestyle='--', alpha=0.85)

ax.set_xlim(-20, 520)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Word Count')
ax.set_ylabel('Spam Word Ratio')
ax.set_title('Five Valid Separating Hyperplanes\n(All classify correctly — but which is BEST?)')
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

print("All 5 lines correctly separate spam from legitimate email.")
print("But they sit at very different distances from the data points.")
print("→ We need a principled way to pick the BEST one.")

## 4. Computing the Margin for Each Hyperplane

For each candidate line, we compute the margin — the minimum distance from the line to any training point.

**Point-to-hyperplane distance formula:**

$$d_i = \frac{|\mathbf{w} \cdot \mathbf{x}_i + b|}{\|\mathbf{w}\|}$$

The **margin** of a hyperplane is:

$$\text{Margin} = 2 \times \min_i \left( \frac{y_i (\mathbf{w} \cdot \mathbf{x}_i + b)}{\|\mathbf{w}\|} \right)$$

We multiply by $y_i$ to ensure we're measuring the **signed** distance — correctly classified points have positive signed distance from the boundary.

A line in 2D with slope $m$ and intercept $c$ can be written as:

$$-m \cdot x_1 + x_2 - c = 0$$

So $\mathbf{w} = (-m, 1)$ and $b = -c$.

In [ ]:
# ============================================================
# CELL 5: Compute the functional margin for each hyperplane
# ============================================================
def compute_margin(m_slope, c_intercept, X_data, y_labels):
    """
    Compute the geometric margin of the line:
       -m * x1 + x2 - c = 0
    i.e., w = [-m_slope, 1], b = -c_intercept

    Returns:
        margin      : scalar, the geometric margin (twice the min distance)
        min_dist    : scalar, minimum distance from any point to the line
        support_idx : index of the closest point (support vector candidate)
    """
    w = np.array([-m_slope, 1.0])   # normal vector to the line
    b = -c_intercept                 # bias term
    w_norm = np.linalg.norm(w)       # ||w||

    # Signed distances: yi * (w·xi + b) / ||w||
    signed_dists = y_labels * (X_data @ w + b) / w_norm

    min_signed_dist = signed_dists.min()   # minimum over all points
    margin = 2 * min_signed_dist           # total margin width
    support_idx = np.argmin(signed_dists)  # index of closest point

    return margin, min_signed_dist, support_idx

# Compute margin for each candidate hyperplane
print(f"{'Hyperplane':<30} {'Margin Width':>14} {'Closest Point Idx':>18}")
print("-" * 65)

margins = []
for hp in candidate_hyperplanes:
    margin, min_d, sv_idx = compute_margin(hp['m'], hp['c'], X, y)
    margins.append(margin)
    print(f"{hp['label']:<30} {margin:>14.4f} {sv_idx:>18d}")

best_idx = np.argmax(margins)
print(f"\nLargest margin: {candidate_hyperplanes[best_idx]['label']} with margin = {margins[best_idx]:.4f}")

## 5. Finding the True Maximum Margin Hyperplane

Instead of guessing from 5 candidates, we can find the **exact** maximum margin hyperplane using scikit-learn's SVC with a linear kernel and a very large penalty $C$ (effectively enforcing hard margins).

The optimal hyperplane satisfies:
- $\mathbf{w} \cdot \mathbf{x} + b = 0$ is the decision boundary
- $\mathbf{w} \cdot \mathbf{x} + b = +1$ is the positive margin line
- $\mathbf{w} \cdot \mathbf{x} + b = -1$ is the negative margin line
- Margin $= 2 / \|\mathbf{w}\|$

In [ ]:
# ============================================================
# CELL 6: Fit the maximum margin classifier using sklearn SVC
# ============================================================
# Scale features so word_count and spam_word_ratio are on similar scales
# This is important for the geometry to be meaningful
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit SVC with linear kernel — large C enforces hard margin behavior
svm = SVC(kernel='linear', C=1e6, random_state=42)
svm.fit(X_scaled, y)

# Extract the learned parameters
w_opt = svm.coef_[0]            # weight vector [w1, w2]
b_opt = svm.intercept_[0]       # bias term b
w_norm = np.linalg.norm(w_opt)  # ||w||
margin_width = 2.0 / w_norm     # margin = 2 / ||w||

# Identify support vectors — the training points on the margin lines
sv_indices = svm.support_          # indices of support vectors in training set
n_sv = len(sv_indices)

print("Maximum Margin Hyperplane (in scaled space):")
print(f"  Weight vector w  = {w_opt}")
print(f"  Bias b           = {b_opt:.4f}")
print(f"  ||w||            = {w_norm:.4f}")
print(f"  Margin width     = 2 / ||w|| = {margin_width:.4f}")
print(f"  Number of SVs    = {n_sv}")
print(f"  SV indices       = {sv_indices}")
print(f"  Train accuracy   = {svm.score(X_scaled, y):.4f}")

In [ ]:
# ============================================================
# CELL 7: Visualize the maximum margin hyperplane
# ============================================================
fig, ax = plt.subplots(figsize=(11, 8))

# --- Scatter plot of data ---
ax.scatter(X_scaled[y ==  1, 0], X_scaled[y ==  1, 1],
           color='crimson', marker='o', s=60, alpha=0.65,
           label='Spam (+1)', edgecolors='darkred', linewidths=0.5, zorder=3)
ax.scatter(X_scaled[y == -1, 0], X_scaled[y == -1, 1],
           color='steelblue', marker='s', s=60, alpha=0.65,
           label='Legitimate (-1)', edgecolors='navy', linewidths=0.5, zorder=3)

# --- Generate the decision boundary and margin lines ---
x1_range = np.linspace(X_scaled[:, 0].min() - 0.3, X_scaled[:, 0].max() + 0.3, 300)

# Decision boundary: w[0]*x1 + w[1]*x2 + b = 0  =>  x2 = (-w[0]*x1 - b) / w[1]
x2_boundary = (-w_opt[0] * x1_range - b_opt) / w_opt[1]

# Positive margin: w·x + b = +1
x2_pos_margin = (-w_opt[0] * x1_range - b_opt + 1) / w_opt[1]

# Negative margin: w·x + b = -1
x2_neg_margin = (-w_opt[0] * x1_range - b_opt - 1) / w_opt[1]

ax.plot(x1_range, x2_boundary, color='black', linewidth=2.5,
        label='Decision Boundary (w·x+b=0)', zorder=4)
ax.plot(x1_range, x2_pos_margin, color='green', linewidth=1.8,
        linestyle='--', label='Margin line: w·x+b=+1', zorder=4)
ax.plot(x1_range, x2_neg_margin, color='green', linewidth=1.8,
        linestyle='--', label='Margin line: w·x+b=-1', zorder=4)

# --- Shade the margin region ---
ax.fill_between(x1_range, x2_neg_margin, x2_pos_margin,
                color='gold', alpha=0.25, label='Margin region', zorder=2)

# --- Highlight support vectors ---
ax.scatter(X_scaled[sv_indices, 0], X_scaled[sv_indices, 1],
           s=220, facecolors='none', edgecolors='limegreen',
           linewidths=2.5, label='Support Vectors', zorder=5)

ax.set_xlim(X_scaled[:, 0].min() - 0.3, X_scaled[:, 0].max() + 0.3)
ax.set_xlabel('Word Count (scaled)')
ax.set_ylabel('Spam Word Ratio (scaled)')
ax.set_title('Maximum Margin Hyperplane\nDecision Boundary + Margin Lines + Support Vectors')
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

## 6. Identifying Support Vectors

**Support vectors** are the training points that lie **exactly on the margin lines** — i.e., the points for which:

$$y_i (\mathbf{w} \cdot \mathbf{x}_i + b) = 1$$

These are the only points that matter for defining the decision boundary. If you remove any **non-support-vector** point from the training set and refit the model, nothing changes. But if you remove or move a support vector, the entire boundary shifts.

This is the remarkable **sparsity property** of SVMs: the decision boundary is determined by only a small subset of the training data.

In [ ]:
# ============================================================
# CELL 8: Verify support vectors and margin distances
# ============================================================
# Compute the functional margin yi*(w·xi + b) for every training point
# Points with functional margin = 1 are support vectors
functional_margins = y * (X_scaled @ w_opt + b_opt)

# Compute geometric distance: yi * (w·xi + b) / ||w||
geometric_distances = functional_margins / w_norm

# Support vectors have functional margin ≈ 1
# (small tolerance for floating point)
tol = 1e-3
sv_mask_manual = np.abs(functional_margins - 1.0) < tol

print("Support Vector Analysis:")
print(f"  Total training points          : {len(X_scaled)}")
print(f"  Support vectors (sklearn)      : {n_sv}")
print(f"  Support vectors (manual check) : {sv_mask_manual.sum()}")
print()
print("Functional margin for each support vector (should be ≈ 1.0):")
for idx in sv_indices:
    class_name = 'Spam' if y[idx] == 1 else 'Legitimate'
    print(f"  Point {idx:3d} [{class_name:>10s}]  "
          f"functional margin = {functional_margins[idx]:.6f}  "
          f"geometric dist = {geometric_distances[idx]:.6f}")

print()
print("Non-support vectors have functional margin > 1 (they are farther away):")
non_sv_mask = np.ones(len(X_scaled), dtype=bool)
non_sv_mask[sv_indices] = False
print(f"  Min functional margin (non-SVs): {functional_margins[non_sv_mask].min():.4f}")
print(f"  All non-SVs satisfy margin ≥ 1 : {(functional_margins[non_sv_mask] >= 1.0 - tol).all()}")

In [ ]:
# ============================================================
# CELL 9: Visualize distance distribution from the boundary
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: histogram of geometric distances
ax = axes[0]
ax.hist(geometric_distances[y == 1], bins=20, color='crimson', alpha=0.6,
        label='Spam (+1)', edgecolor='darkred')
ax.hist(geometric_distances[y == -1], bins=20, color='steelblue', alpha=0.6,
        label='Legitimate (-1)', edgecolor='navy')

# Mark the margin at distance = 1/||w|| from boundary
margin_dist = 1.0 / w_norm
ax.axvline(margin_dist, color='green', linewidth=2, linestyle='--',
           label=f'Margin boundary = {margin_dist:.3f}')
ax.axvline(-margin_dist, color='green', linewidth=2, linestyle='--')
ax.axvline(0, color='black', linewidth=2, label='Decision boundary')

ax.set_xlabel('Geometric Distance from Boundary')
ax.set_ylabel('Count')
ax.set_title('Distribution of Point Distances\nfrom the Decision Boundary')
ax.legend(fontsize=10)

# Right: bar chart showing functional margins for all points
ax2 = axes[1]
# Sort by functional margin to make it readable
sorted_idx = np.argsort(functional_margins)
colors_bar = ['crimson' if y[i] == 1 else 'steelblue' for i in sorted_idx]
sv_flag = [sv_mask_manual[i] for i in sorted_idx]

# Only show a sample of 40 points for clarity
sample_step = max(1, len(sorted_idx) // 40)
sample_idx = sorted_idx[::sample_step]
ax2.bar(range(len(sample_idx)), functional_margins[sample_idx],
        color=['crimson' if y[i] == 1 else 'steelblue' for i in sample_idx],
        alpha=0.75, edgecolor='gray', linewidth=0.3)
ax2.axhline(1.0, color='green', linewidth=2, linestyle='--',
            label='Margin = 1 (support vectors)')
ax2.set_xlabel('Email (sorted by functional margin)')
ax2.set_ylabel('Functional Margin  yᵢ(w·xᵢ + b)')
ax2.set_title('Functional Margins for All Training Points\n(Support vectors sit exactly at margin = 1)')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 7. Why Maximum Margin Generalizes Better

Here's the intuitive argument for why a wider margin leads to better generalization:

**Narrow margin = fragile classifier:**
- The boundary sits very close to some training points
- Small shifts in new data → misclassification
- The classifier is "memorizing" the exact training data

**Wide margin = robust classifier:**
- Lots of space between the boundary and any training point
- New points can vary considerably and still land on the correct side
- The classifier has captured the underlying pattern, not just the training noise

**Formal argument (VC theory):**
- The generalization error is bounded by: $\text{Error} \leq \frac{c}{n} \left(\frac{R^2}{\gamma^2} + 1\right)$
- where $\gamma = 1/\|\mathbf{w}\|$ is the margin, $R$ is the radius of the data ball, $n$ is training size
- Larger margin $\gamma$ → smaller bound → better generalization guarantee
- This is a **formal mathematical guarantee** — not just intuition

In [ ]:
# ============================================================
# CELL 10: Demonstrate margin vs. classification robustness
# Compare the maximum margin line against a narrower alternative
# ============================================================
# Create a narrow-margin classifier by fitting SVM on a subset that
# forces it toward one class
np.random.seed(42)

# Perturbed weights: slightly off from optimal (simulating a narrow-margin solution)
# We manually set a non-optimal w and b for illustration
w_narrow  = np.array([2.5, 0.8])    # arbitrary non-optimal weights
b_narrow  = -0.5                     # non-optimal bias
norm_narrow = np.linalg.norm(w_narrow)
margin_narrow = 2.0 / norm_narrow

print("Comparison: Maximum Margin vs. Narrow Margin Hyperplane")
print("=" * 55)
print(f"Maximum margin solution:")
print(f"  ||w||   = {w_norm:.4f}")
print(f"  Margin  = 2 / ||w|| = {margin_width:.4f}")
print()
print(f"Narrow margin solution (non-optimal):")
print(f"  ||w||   = {norm_narrow:.4f}")
print(f"  Margin  = 2 / ||w|| = {margin_narrow:.4f}")
print()
print(f"Maximum margin is {margin_width / margin_narrow:.2f}x wider than narrow margin")
print()

# Verify that the narrow-margin hyperplane still correctly classifies
# (it would need to be checked against actual data; here it's illustrative)
print("Key takeaway:")
print("  A wider margin gives MORE room for new test points to be classified correctly.")
print("  This is not just geometry — it has a formal guarantee via VC dimension theory.")

In [ ]:
# ============================================================
# CELL 11: Visualize the margin formula: Margin = 2 / ||w||
# Show how increasing ||w|| shrinks the margin
# ============================================================
w_norms = np.linspace(0.5, 5.0, 200)   # range of ||w|| values
margins  = 2.0 / w_norms               # corresponding margins

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(w_norms, margins, color='steelblue', linewidth=2.5,
        label=r'Margin $= \frac{2}{\|\mathbf{w}\|}$')

# Mark the optimal ||w|| found by our SVM
ax.axvline(w_norm, color='crimson', linewidth=2, linestyle='--',
           label=f'SVM solution: ||w|| = {w_norm:.3f}')
ax.axhline(margin_width, color='green', linewidth=2, linestyle=':',
           label=f'SVM margin = {margin_width:.3f}')
ax.scatter([w_norm], [margin_width], s=150, color='gold',
           edgecolors='black', linewidths=1.5, zorder=5,
           label='Maximum margin point')

ax.set_xlabel(r'Weight Vector Norm $\|\mathbf{w}\|$')
ax.set_ylabel('Margin Width')
ax.set_title(r'Margin $= \frac{2}{\|\mathbf{w}\|}$ — Larger $\|w\|$ = Smaller Margin'
             '\nSVM minimizes ||w|| to maximize margin')
ax.legend(fontsize=11)
ax.set_xlim(0.5, 5.0)
ax.set_ylim(0, 4.5)
plt.tight_layout()
plt.show()

print("Critical insight: minimizing ||w||² (the SVM objective) = maximizing the margin.")
print("These are two ways of stating the same geometric problem.")

In [ ]:
# ============================================================
# CELL 12: Summarize all key numbers in a tidy table
# ============================================================
summary_data = {
    'Quantity': [
        'Weight vector w',
        'Bias b',
        'Norm ||w||',
        'Margin (2 / ||w||)',
        'Number of Support Vectors',
        'Support Vector Indices',
        'Training Accuracy',
        'Min functional margin (all points)',
        'Max functional margin (all points)',
    ],
    'Value': [
        str(np.round(w_opt, 4)),
        f'{b_opt:.4f}',
        f'{w_norm:.4f}',
        f'{margin_width:.4f}',
        str(n_sv),
        str(sv_indices.tolist()),
        f'{svm.score(X_scaled, y):.4f}',
        f'{functional_margins.min():.4f}',
        f'{functional_margins.max():.4f}',
    ]
}

summary_df = pd.DataFrame(summary_data)
print("Maximum Margin SVM — Key Results Summary")
print("=" * 60)
print(summary_df.to_string(index=False))

print()
print("Note: All functional margins ≥ 1, confirming correct classification")
print(f"Support vectors (functional margin ≈ 1): {sv_indices.tolist()}")

## 8. Putting It All Together: The Geometry of Maximum Margin

Let's create one final comprehensive visualization that ties every concept together:

- The decision boundary
- The margin lines at $\pm 1$
- The shaded margin region
- Support vectors highlighted
- Distance annotations to the margin

In [ ]:
# ============================================================
# CELL 13: Full annotated visualization — the "poster" figure
# ============================================================
fig, ax = plt.subplots(figsize=(12, 9))

# --- Data points ---
ax.scatter(X_scaled[y ==  1, 0], X_scaled[y ==  1, 1],
           color='crimson', marker='o', s=55, alpha=0.65,
           edgecolors='darkred', linewidths=0.5, zorder=3)
ax.scatter(X_scaled[y == -1, 0], X_scaled[y == -1, 1],
           color='steelblue', marker='s', s=55, alpha=0.65,
           edgecolors='navy', linewidths=0.5, zorder=3)

# --- Decision boundary and margin lines ---
x1_plot = np.linspace(X_scaled[:, 0].min() - 0.3, X_scaled[:, 0].max() + 0.3, 300)
x2_db   = (-w_opt[0] * x1_plot - b_opt) / w_opt[1]
x2_pos  = (-w_opt[0] * x1_plot - b_opt + 1) / w_opt[1]
x2_neg  = (-w_opt[0] * x1_plot - b_opt - 1) / w_opt[1]

ax.plot(x1_plot, x2_db,  color='black',  lw=2.5, label='Decision Boundary: w·x+b=0')
ax.plot(x1_plot, x2_pos, color='green',  lw=1.8, ls='--', label='Margin: w·x+b=+1 (spam side)')
ax.plot(x1_plot, x2_neg, color='green',  lw=1.8, ls='--', label='Margin: w·x+b=−1 (legit side)')
ax.fill_between(x1_plot, x2_neg, x2_pos, color='limegreen', alpha=0.12, label='Margin region')

# --- Highlight support vectors ---
ax.scatter(X_scaled[sv_indices, 0], X_scaled[sv_indices, 1],
           s=260, facecolors='none', edgecolors='gold',
           linewidths=2.8, zorder=5, label='Support Vectors')

# --- Annotate one support vector with distance to boundary ---
sv0 = sv_indices[0]
x_sv  = X_scaled[sv0]
dist0 = 1.0 / w_norm  # half-margin
ax.annotate(
    f'Support Vector\ndist = {dist0:.3f}',
    xy=(x_sv[0], x_sv[1]),
    xytext=(x_sv[0] + 0.6, x_sv[1] - 0.4),
    fontsize=10, color='darkgreen',
    arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.5)
)

# --- Labels and legend ---
ax.text(0.02, 0.97, 'SPAM', transform=ax.transAxes,
        fontsize=13, color='crimson', fontweight='bold', va='top')
ax.text(0.02, 0.05, 'LEGITIMATE', transform=ax.transAxes,
        fontsize=13, color='steelblue', fontweight='bold')

# Custom legend entries for data points
legend_elements = [
    mpatches.Patch(facecolor='crimson', edgecolor='darkred', label='Spam (+1)'),
    mpatches.Patch(facecolor='steelblue', edgecolor='navy', label='Legitimate (−1)'),
    Line2D([0], [0], color='black', lw=2.5, label='Decision Boundary'),
    Line2D([0], [0], color='green', lw=1.8, ls='--', label='Margin lines (±1)'),
    mpatches.Patch(facecolor='limegreen', alpha=0.3, label='Margin region'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
           markeredgecolor='gold', markersize=12, markeredgewidth=2.5,
           label='Support Vectors'),
]

ax.set_xlim(X_scaled[:, 0].min() - 0.3, X_scaled[:, 0].max() + 0.3)
ax.set_xlabel('Word Count (scaled)', fontsize=13)
ax.set_ylabel('Spam Word Ratio (scaled)', fontsize=13)
ax.set_title(
    f'Maximum Margin Hyperplane — Email Spam Classification\n'
    f'Margin = 2/||w|| = {margin_width:.4f}  |  {n_sv} Support Vectors  |  '
    f'200 training emails',
    fontsize=13
)
ax.legend(handles=legend_elements, fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

## 9. Key Takeaways

| Concept | Summary |
|---|---|
| **Linear separability** | Data can be perfectly split by a hyperplane |
| **Hyperplane equation** | $\mathbf{w} \cdot \mathbf{x} + b = 0$ — classifies with sign |
| **Distance formula** | $d = \|\mathbf{w} \cdot \mathbf{x} + b\| / \|\mathbf{w}\|$ |
| **Margin** | $2 / \|\mathbf{w}\|$ — the gap between the two classes |
| **Maximum margin** | Unique hyperplane that maximizes the margin |
| **Support vectors** | Points that lie exactly ON the margin lines |
| **Sparsity** | Only support vectors define the boundary — most training points are irrelevant |
| **Generalization** | Larger margin → better VC bound → better generalization |

**What comes next:**
- Notebook 02 formalizes this as an optimization problem (the Hard-Margin SVM primal)
- Notebook 03 relaxes the separability assumption (Soft-Margin SVM)
- Notebook 04 introduces kernel methods for non-linear boundaries

---

## Self-Check Questions

### Question 1
Five different hyperplanes all correctly classify a linearly separable dataset. What makes the maximum margin hyperplane **special** among all valid separating hyperplanes?

<details>
<summary>Click to reveal answer</summary>

The maximum margin hyperplane is special because it **maximizes the minimum distance from the decision boundary to any training point**. Formally, it is the unique hyperplane that:

1. Correctly classifies all training points
2. Maximizes the margin $2 / \|\mathbf{w}\|$ (the perpendicular gap between the two class boundaries)

All other valid separating hyperplanes have a smaller margin — they sit closer to at least one training point. The maximum margin hyperplane is special because:
- It is **unique** (only one hyperplane achieves the maximum margin, due to the convexity of the problem)
- It has the **best generalization guarantee** (via VC theory: larger margin → smaller complexity → lower test error bound)
- It is **robust**: new test points that vary somewhat from training data are less likely to be misclassified when the margin is large

Think of it as the ruler that is equidistant from both groups of M&Ms — not just anywhere in the gap, but exactly in the middle.

</details>

### Question 2
If you remove a **non-support-vector** point from the training set, does the decision boundary change? What if you remove a **support vector**?

<details>
<summary>Click to reveal answer</summary>

**Removing a non-support-vector:** The decision boundary does **NOT** change.

Non-support-vectors are points that have a functional margin strictly greater than 1 — they lie comfortably on the correct side of the margin line, far from the boundary. The maximum margin hyperplane is determined **entirely** by the support vectors. Removing any other point leaves the optimization problem with the same constraints that matter (the active constraints, i.e., the support vectors), so the solution is identical.

**Removing a support vector:** The decision boundary **WILL** change.

Support vectors are the "critical" points — they sit exactly on the margin lines and define where the margin is. Removing one of them changes the set of binding constraints. The new optimal hyperplane will have a different (larger) margin, since there are fewer tight constraints. The boundary shifts to accommodate the new maximum margin.

This asymmetry is the remarkable **sparsity property** of SVMs: the entire model is determined by only a handful of support vectors, not the full training set.

</details>

### Question 3
The margin is $2 / \|\mathbf{w}\|$. If you **double** $\|\mathbf{w}\|$, what happens to the margin? What happens to the decision boundary $\mathbf{w} \cdot \mathbf{x} + b = 0$?

<details>
<summary>Click to reveal answer</summary>

**Effect on the margin:** If you double $\|\mathbf{w}\|$, the margin is halved:

$$\text{New margin} = \frac{2}{2\|\mathbf{w}\|} = \frac{1}{\|\mathbf{w}\|} = \frac{1}{2} \times \text{Original margin}$$

**Effect on the decision boundary:** The decision boundary $\mathbf{w} \cdot \mathbf{x} + b = 0$ does **NOT** change geometrically.

Here is why: if you replace $\mathbf{w}$ with $2\mathbf{w}$ and $b$ with $2b$, the equation $2\mathbf{w} \cdot \mathbf{x} + 2b = 0$ is identical to $\mathbf{w} \cdot \mathbf{x} + b = 0$ (just multiplied through by 2). The same set of points $\mathbf{x}$ satisfies both equations — the hyperplane is unchanged.

However, the **margin lines** $\mathbf{w} \cdot \mathbf{x} + b = \pm 1$ **do** change: with $2\mathbf{w}$, the margin lines are $2\mathbf{w} \cdot \mathbf{x} + 2b = \pm 1$, which corresponds to $\mathbf{w} \cdot \mathbf{x} + b = \pm 0.5$. These are closer to the decision boundary — the margin is narrower.

**Practical implication:** The scale of $\|\mathbf{w}\|$ is not arbitrary in SVMs. The constraint $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$ pins the margin lines at functional margin $\pm 1$. Minimizing $\|\mathbf{w}\|$ subject to these constraints forces the widest possible margin — the two are mathematically equivalent.

</details>